# Weighted Ensemble

A stronger ADME/Tox-style ensemble that combines multiple tabular models and learns non-negative blending weights on the validation split.

In [1]:
from pathlib import Path
import json
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.optimize import minimize
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNetCV, RidgeCV
from sklearn.metrics import mean_absolute_error

PROJECT_ROOT = Path('../..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.modeling import artifact_paths, load_tabular_arrays, plot_pred_vs_real, regression_metrics

In [ ]:
RUN_ID = 'weighted_ensemble'
RANDOM_STATE = 42

#load data
X_train, X_valid, X_test, y_train, y_valid, y_test, feature_names = load_tabular_arrays()
X_train = X_train.astype(np.float32)
X_valid = X_valid.astype(np.float32)
X_test = X_test.astype(np.float32)

#output paths where to find things!
paths = artifact_paths(Path.cwd(), RUN_ID, '.joblib', include_metadata=True)
paths['weights'] = Path.cwd() / 'outcome' / 'metadata' / f'{RUN_ID}_weights.csv'
paths['base_predictions'] = Path.cwd() / 'outcome' / 'predictions' / f'{RUN_ID}_base_predictions.csv'
for path in paths.values():
    path.parent.mkdir(parents=True, exist_ok=True)

#to visualize the data distribution
print(f'Train: {X_train.shape}, valid: {X_valid.shape}, test: {X_test.shape}')

Train: (5170, 1813), valid: (738, 1813), test: (1477, 1813)


In [ ]:
#model preparation
def make_estimators():
    return {
        'ridge': RidgeCV(alphas=np.logspace(-3, 3, 13)),
        'elastic_net': ElasticNetCV(
            l1_ratio=[0.05, 0.1, 0.25, 0.5, 0.8],
            alphas=np.logspace(-4, 1, 12),
            cv=5,
            max_iter=20_000,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
        'random_forest': RandomForestRegressor(
            n_estimators=600,
            max_features='sqrt',
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
        'extra_trees': ExtraTreesRegressor(
            n_estimators=800,
            max_features='sqrt',
            min_samples_leaf=1,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
        'hist_gradient_boosting': HistGradientBoostingRegressor(
            learning_rate=0.04,
            max_iter=700,
            max_leaf_nodes=31,
            l2_regularization=0.05,
            early_stopping=True,
            random_state=RANDOM_STATE,
        ),
        'xgboost': xgb.XGBRegressor(
            n_estimators=800,
            learning_rate=0.035,
            max_depth=5,
            min_child_weight=2,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.05,
            reg_lambda=1.5,
            objective='reg:squarederror',
            tree_method='hist',
            device='cuda',
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    }

#model management alligned
def fit_estimator(name, estimator, X_fit, y_fit, X_eval=None, y_eval=None):
    if name == 'xgboost' and X_eval is not None:
        try:
            estimator.fit(X_fit, y_fit, eval_set=[(X_eval, y_eval)], verbose=False)
        except Exception as exc:
            warnings.warn(f'XGBoost CUDA failed; retrying on CPU. Details: {exc}')
            estimator.set_params(device='cpu')
            estimator.fit(X_fit, y_fit, eval_set=[(X_eval, y_eval)], verbose=False)
    else:
        estimator.fit(X_fit, y_fit)
    return estimator

In [ ]:
#train base models and collect predictions for the ensemble weights optimization
base_estimators = make_estimators()
fitted_for_weights = {}
valid_predictions = {}

for name, estimator in base_estimators.items():
    print(f'Training {name}...')
    fitted = fit_estimator(name, clone(estimator), X_train, y_train, X_valid, y_valid)
    preds = np.asarray(fitted.predict(X_valid)).ravel()
    fitted_for_weights[name] = fitted
    valid_predictions[name] = preds
    metrics = regression_metrics(y_valid, preds)
    print(f"  valid MAE={metrics['mae']:.4f} RMSE={metrics['rmse']:.4f} R2={metrics['r2']:.4f}")

valid_pred_matrix = np.column_stack([valid_predictions[name] for name in base_estimators])

Training ridge...
  valid MAE=0.5208 RMSE=0.7125 R2=0.4695
Training elastic_net...


/home/luca/miniconda3/miniconda3/envs/tox_prediction/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.273e+00, tolerance: 3.769e-01
  model = cd_fast.enet_coordinate_descent_gram(
/home/luca/miniconda3/miniconda3/envs/tox_prediction/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.835e+00, tolerance: 3.475e-01
  model = cd_fast.enet_coordinate_descent_gram(
/home/luca/miniconda3/miniconda3/envs/tox_prediction/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase th

  valid MAE=0.5243 RMSE=0.7157 R2=0.4646
Training random_forest...
  valid MAE=0.4624 RMSE=0.6396 R2=0.5725
Training extra_trees...
  valid MAE=0.4402 RMSE=0.6127 R2=0.6076
Training hist_gradient_boosting...
  valid MAE=0.4348 RMSE=0.6038 R2=0.6189
Training xgboost...


/home/luca/miniconda3/miniconda3/envs/tox_prediction/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [08:14:01] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1772125046773/work/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


  valid MAE=0.4417 RMSE=0.6117 R2=0.6089


In [ ]:
#optimize ensemble weights using constrained optimization
def weighted_prediction(weights, pred_matrix):
    weights = np.asarray(weights, dtype=float)
    return pred_matrix @ weights


#
def objective(weights):
    return mean_absolute_error(y_valid, weighted_prediction(weights, valid_pred_matrix))


n_models = valid_pred_matrix.shape[1]
initial_weights = np.full(n_models, 1.0 / n_models)
constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
bounds = [(0.0, 1.0)] * n_models

result = minimize(
    objective,
    initial_weights,
    method='SLSQP',
    bounds=bounds,
    constraints=constraints,
    options={'maxiter': 1000, 'ftol': 1e-10},
)

if result.success:
    weights = result.x
else:
    warnings.warn(f'Weight optimization failed: {result.message}. Falling back to equal weights.')
    weights = initial_weights

weights = weights / weights.sum()
weight_table = pd.DataFrame({'model': list(base_estimators), 'weight': weights})
display(weight_table.sort_values('weight', ascending=False))

valid_ensemble = weighted_prediction(weights, valid_pred_matrix)
regression_metrics(y_valid, valid_ensemble)

,model,weight
4,hist_gradient_boosting,5.387961e-01
3,extra_trees,4.612039e-01
1,elastic_net,6.172802e-17
2,random_forest,4.392766e-17
5,xgboost,2.557158e-18
0,ridge,1.907541e-19


{'rmse': 0.5988616793228617,
 'mae': 0.4300907328214799,
 'r2': 0.6251753180484347}

In [ ]:
#final fit on train + validation, then evaluate once on the held-out test set.
X_full = np.vstack([X_train, X_valid])
y_full = np.concatenate([y_train, y_valid])

final_estimators = {}
test_predictions = {}

for name, estimator in make_estimators().items():
    print(f'Final training {name}...')
    final_model = fit_estimator(name, clone(estimator), X_full, y_full, X_valid, y_valid)
    final_estimators[name] = final_model
    test_predictions[name] = np.asarray(final_model.predict(X_test)).ravel()

test_pred_matrix = np.column_stack([test_predictions[name] for name in base_estimators])
ensemble_predictions = weighted_prediction(weights, test_pred_matrix)
test_metrics = regression_metrics(y_test, ensemble_predictions)
print(f"[Weighted Ensemble] RMSE: {test_metrics['rmse']:.4f} | MAE: {test_metrics['mae']:.4f} | R2: {test_metrics['r2']:.4f}")
test_metrics

Final training ridge...
Final training elastic_net...


/home/luca/miniconda3/miniconda3/envs/tox_prediction/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.554e+00, tolerance: 4.478e-01
  model = cd_fast.enet_coordinate_descent_gram(
/home/luca/miniconda3/miniconda3/envs/tox_prediction/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.588e+00, tolerance: 4.203e-01
  model = cd_fast.enet_coordinate_descent_gram(
/home/luca/miniconda3/miniconda3/envs/tox_prediction/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase th

Final training random_forest...
Final training extra_trees...
Final training hist_gradient_boosting...
Final training xgboost...


/home/luca/miniconda3/miniconda3/envs/tox_prediction/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [08:25:42] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1772125046773/work/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


[Weighted Ensemble] RMSE: 0.5553 | MAE: 0.4118 | R2: 0.6548


{'rmse': 0.5553029877895395,
 'mae': 0.41178118659615504,
 'r2': 0.6548482411363561}

In [ ]:
#save everything for analysis and reproducibility
base_prediction_df = pd.DataFrame({'real': y_test, 'ensemble': ensemble_predictions})
for name in base_estimators:
    base_prediction_df[name] = test_predictions[name]

base_prediction_df[['real', 'ensemble']].rename(columns={'ensemble': 'prediction'}).to_csv(paths['predictions'], index=False)
base_prediction_df.to_csv(paths['base_predictions'], index=False)
weight_table.to_csv(paths['weights'], index=False)
plot_pred_vs_real(paths['pred_vs_real'], y_test, ensemble_predictions, 'Weighted Ensemble: Predicted vs Real')

#joblib.dump({'models': final_estimators, 'weights': dict(zip(base_estimators.keys(), weights)), 'feature_names': feature_names}, paths['model'])

metadata = {
    'run_id': RUN_ID,
    'metrics': test_metrics,
    'validation_metrics': regression_metrics(y_valid, valid_ensemble),
    'weights': dict(zip(base_estimators.keys(), map(float, weights))),
}
with paths['metadata'].open('w', encoding='utf-8') as fp:
    json.dump(metadata, fp, indent=2)

paths

{'model': PosixPath('/home/luca/Documents/SUPSI/toxicity-prediction/notebooks/ml-models/outcome/models/weighted_ensemble_model.joblib'),
 'predictions': PosixPath('/home/luca/Documents/SUPSI/toxicity-prediction/notebooks/ml-models/outcome/predictions/weighted_ensemble_predictions.csv'),
 'pred_vs_real': PosixPath('/home/luca/Documents/SUPSI/toxicity-prediction/notebooks/ml-models/outcome/weighted_ensemble_pred_vs_real.png'),
 'metadata': PosixPath('/home/luca/Documents/SUPSI/toxicity-prediction/notebooks/ml-models/outcome/metadata/weighted_ensemble_metadata.json'),
 'weights': PosixPath('/home/luca/Documents/SUPSI/toxicity-prediction/notebooks/ml-models/outcome/metadata/weighted_ensemble_weights.csv'),
 'base_predictions': PosixPath('/home/luca/Documents/SUPSI/toxicity-prediction/notebooks/ml-models/outcome/predictions/weighted_ensemble_base_predictions.csv')}